# **Session 7**

## Data Visualisation with Plots

In Session 6 we described the data using **numbers** (mean, median, std, counts).
Today we turn those numbers into **pictures**. A good plot can show patterns, outliers and relationships very quickly.

For every plot we will ask three simple questions:

1. **What does it show?**
2. **When do we use it?**
3. **How do we make it in Python?**

### Plan for today

| Part | Topic |
|---|---|
| 1 | Recap and setup |
| 2 | Box plots |
| 3 | Histograms |
| 4 | Count plots and bar charts |
| 5 | Line plots |
| 6 | Scatter plots |
| 7 | Pie charts |
| 8 | Pair plot and correlation heatmap |
| 9 | Choosing the right plot |
| 10 | Wrap-up and practice |

## Part 1: Recap and Setup

First we load the data, exactly like before.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls "/content/drive/MyDrive/Colab Notebooks/Data/"

import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Data/academic_data.csv")

### Import the plotting libraries

| Library | Short name | What it does |
|---|---|---|
| `matplotlib.pyplot` | `plt` | The main plotting library. Titles, labels and showing the plot. |
| `seaborn` | `sns` | Built on matplotlib. It gives us nicer plots like pair plot and heatmap with one line of code. |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### Quick recap of Session 6 (cleaning)

We make a copy of the data and fix the two small problems we found: the empty `ai_tool` cells and the dash in `complaint_type`.

In [ ]:
df_clean = df.copy()

df_clean["ai_tool"] = df_clean["ai_tool"].fillna("No Tool")
df_clean["complaint_type"] = df_clean["complaint_type"].replace("-", "No Complaint")

df_clean.head()

### Column lists

We store the names of the continuous columns in a list, so we can use them again and again.

In [ ]:
continuous_cols = ["age", "study_hours", "attendance", "ai_usage_hours", "marks"]

### Every good plot needs 3 things

1. A **title** (what is this plot about?)
2. An **x-axis label**
3. A **y-axis label**

We will add them to every plot using `plt.title()`, `plt.xlabel()` and `plt.ylabel()`. At the end of each plot we write `plt.show()` to display it.

## Part 2: Box Plots

### What does it show?

A box plot shows the **spread of a continuous variable** using 5 numbers: minimum, Q1, median, Q3 and maximum. It also shows unusual values (**outliers**).

### How to read it

| Part of the plot | Meaning |
|---|---|
| The **box** | Middle 50 percent of the data (from Q1 to Q3) |
| The **line inside** the box | Median |
| The **whiskers** (lines going out) | Smallest and largest normal values |
| **Dots** beyond the whiskers | Outliers (very unusual values) |

### When do we use it?

- To see the spread and the middle of one variable.
- To spot outliers.
- To **compare** a variable across groups (for example marks of different grades).

### 2.1 Box plot of one variable (`marks`)

In [ ]:
df_clean["marks"].plot(kind="box")
plt.title("Box Plot of Marks")
plt.ylabel("Marks")
plt.show()

**Explanation**

- `df_clean["marks"]` selects the column.
- `.plot(kind="box")` draws the box plot.
- Look at the line in the middle of the box. That is the median marks.

### 2.2 Box plots of all continuous variables

If we put all variables in one plot, the small ones (like `age`) look flat because `attendance` and `marks` are much bigger. So we draw each variable in its **own small plot**.

In [ ]:
df_clean[continuous_cols].plot(kind="box", subplots=True, layout=(1, 5), figsize=(15, 4))
plt.suptitle("Box Plots of All Continuous Variables")
plt.show()

**Explanation**

- `df_clean[continuous_cols]` selects all five continuous columns.
- `subplots=True` gives each column its own small plot.
- `layout=(1, 5)` means 1 row and 5 columns of plots.
- `figsize=(15, 4)` sets the size (width 15, height 4).
- `plt.suptitle()` adds one big title for the whole figure.

### 2.3 Box plots grouped by a category

Now we compare groups. For example: how are marks spread for each grade?

In [ ]:
df_clean.boxplot(column="marks", by="grade")
plt.suptitle("")
plt.title("Marks by Grade")
plt.xlabel("Grade")
plt.ylabel("Marks")
plt.show()

**Explanation**

- `df_clean.boxplot(column="marks", by="grade")` draws one box for every grade.
- `plt.suptitle("")` removes the extra automatic title that pandas adds.
- Grade A should be at the top with the highest marks. If not, something is wrong with the data.

In [ ]:
df_clean.boxplot(column="study_hours", by="section")
plt.suptitle("")
plt.title("Study Hours by Section")
plt.xlabel("Section")
plt.ylabel("Study Hours")
plt.show()

**What to look for**

- Is the **median line** higher for one group?
- Is one box **taller** (more spread) than the others?
- Do the boxes **overlap** a lot? If yes, the groups are not very different.

### 2.4 Reading outliers

Dots outside the whiskers are possible outliers. Let us find the actual numbers using Q1, Q3 and the 1.5 x IQR rule (the same rule the box plot uses).

In [ ]:
q1 = df_clean["marks"].quantile(0.25)
q3 = df_clean["marks"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1    :", q1)
print("Q3    :", q3)
print("IQR   :", iqr)
print("Lower limit:", lower)
print("Upper limit:", upper)

In [ ]:
df_clean[(df_clean["marks"] < lower) | (df_clean["marks"] > upper)]

**Explanation**

- `quantile(0.25)` gives Q1 and `quantile(0.75)` gives Q3.
- IQR = Q3 - Q1.
- Any value below `lower` or above `upper` is an outlier.
- In the last line, `|` means **OR**. We show the students whose marks are below the lower limit or above the upper limit.
- If the table is empty, there are no outliers in `marks`.

## Part 3: Histograms

### What does it show?

A histogram shows the **shape of one continuous variable**. It divides the values into ranges (called **bins**) and shows how many students fall in each range.

### When do we use it?

- To see where most values are.
- To see if the data is balanced or tilted to one side.

### How to read the shape

| Shape | Meaning | Mean vs Median |
|---|---|---|
| **Symmetric** | Balanced like a bell | Mean is about equal to median |
| **Right-skewed** | Long tail on the right (a few very high values) | Mean is bigger than median |
| **Left-skewed** | Long tail on the left (a few very low values) | Mean is smaller than median |

In [ ]:
df_clean["marks"].plot(kind="hist", bins=10)
plt.title("Histogram of Marks")
plt.xlabel("Marks")
plt.ylabel("Number of Students")
plt.show()

**Explanation**

- `kind="hist"` draws the histogram.
- `bins=10` means 10 bars. Each bar is one range of marks.
- A taller bar means more students in that range.

### 3.1 Choosing the number of bins

The number of bins changes how the plot looks. Let us try 5, 10 and 20.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_clean["marks"].plot(kind="hist", bins=5, ax=axes[0])
axes[0].set_title("5 bins")

df_clean["marks"].plot(kind="hist", bins=10, ax=axes[1])
axes[1].set_title("10 bins")

df_clean["marks"].plot(kind="hist", bins=20, ax=axes[2])
axes[2].set_title("20 bins")

plt.show()

**Explanation**

- `plt.subplots(1, 3)` makes 3 small plot areas in one row. We call them `axes[0]`, `axes[1]` and `axes[2]`.
- `ax=axes[0]` tells pandas which small area to draw on.
- `set_title()` gives a title to one small plot.

**Lesson:** too few bins hide the details, too many bins make the plot messy. Around 10 is usually a good start for 100 students.

### 3.2 Mean and median on the histogram

Let us draw the mean and the median as lines.

In [ ]:
df_clean["marks"].plot(kind="hist", bins=10)

plt.axvline(df_clean["marks"].mean(), color="red", linestyle="--", label="Mean")
plt.axvline(df_clean["marks"].median(), color="green", label="Median")

plt.title("Marks with Mean and Median")
plt.xlabel("Marks")
plt.ylabel("Number of Students")
plt.legend()
plt.show()

**Explanation**

- `plt.axvline(x)` draws a vertical line at the value `x`.
- `label=` gives the name that appears in the legend, and `plt.legend()` shows the legend.
- If the red and green lines are close to each other, the data is roughly balanced.

### 3.3 Comparing two histograms side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_clean["study_hours"].plot(kind="hist", bins=10, ax=axes[0])
axes[0].set_title("Study Hours")
axes[0].set_xlabel("Hours")
axes[0].set_ylabel("Number of Students")

df_clean["attendance"].plot(kind="hist", bins=10, ax=axes[1])
axes[1].set_title("Attendance")
axes[1].set_xlabel("Attendance")
axes[1].set_ylabel("Number of Students")

plt.show()

**Explanation:** The left plot shows study hours and the right plot shows attendance. Which one has most students on the higher side?

## Part 4: Count Plots and Bar Charts

These two look similar, but they answer different questions.

| Plot | Question it answers | Example |
|---|---|---|
| **Count plot** | How **many** students are in each category? | Number of students in each section |
| **Bar chart** | What is the **average (or total)** of a number for each category? | Average marks for each section |

### 4.1 Count plot

We count the categories using `value_counts()` and then draw bars.

In [ ]:
df_clean["section"].value_counts().plot(kind="bar")
plt.title("Number of Students in Each Section")
plt.xlabel("Section")
plt.ylabel("Number of Students")
plt.xticks(rotation=0)
plt.show()

**Explanation**

- `value_counts()` counts the students in each section.
- `.plot(kind="bar")` draws one bar for each section.
- `plt.xticks(rotation=0)` keeps the labels on the x-axis straight (not tilted).

In [ ]:
df_clean["ai_tool"].value_counts().plot(kind="bar")
plt.title("Number of Students for Each AI Tool")
plt.xlabel("AI Tool")
plt.ylabel("Number of Students")
plt.xticks(rotation=30)
plt.show()

In [ ]:
df_clean["grade"].value_counts().sort_index().plot(kind="bar")
plt.title("Number of Students for Each Grade")
plt.xlabel("Grade")
plt.ylabel("Number of Students")
plt.xticks(rotation=0)
plt.show()

**Explanation:** `sort_index()` puts the grades in the order A, B, C, D, F. Without it, the bars are sorted from the biggest to the smallest.

### Count plot with seaborn (shorter way)

Seaborn can count and draw in one line.

In [ ]:
sns.countplot(data=df_clean, x="gender")
plt.title("Number of Students by Gender")
plt.xlabel("Gender")
plt.ylabel("Number of Students")
plt.show()

**Explanation:** `sns.countplot(data=df_clean, x="gender")` does the counting and the drawing together. We do not need `value_counts()`.

### 4.2 Bar chart (average of a number for each category)

In [ ]:
df_clean.groupby("section")["marks"].mean().plot(kind="bar")
plt.title("Average Marks by Section")
plt.xlabel("Section")
plt.ylabel("Average Marks")
plt.xticks(rotation=0)
plt.show()

**Explanation**

- `groupby("section")["marks"].mean()` gives the average marks of each section (as we learned in Session 6).
- `.plot(kind="bar")` shows those averages as bars. A taller bar means a higher average.

In [ ]:
df_clean.groupby("ai_tool")["marks"].mean().plot(kind="bar")
plt.title("Average Marks by AI Tool")
plt.xlabel("AI Tool")
plt.ylabel("Average Marks")
plt.xticks(rotation=30)
plt.show()

In [ ]:
df_clean.groupby("year")["study_hours"].mean().plot(kind="bar")
plt.title("Average Study Hours by Year")
plt.xlabel("Year")
plt.ylabel("Average Study Hours")
plt.xticks(rotation=0)
plt.show()

### 4.3 Grouped bar chart

A grouped bar chart has **two categories** at the same time. Example: average marks by grade **and** gender. Each grade gets two bars, one for each gender.

In [ ]:
df_clean.groupby(["grade", "gender"])["marks"].mean()

In [ ]:
df_clean.groupby(["grade", "gender"])["marks"].mean().unstack()

**Explanation**

- `groupby(["grade", "gender"])` makes a group for every combination of grade and gender.
- `.unstack()` moves gender into columns, so we get a neat table with grades in rows and genders in columns. This format is what we need for a grouped bar chart.

In [ ]:
df_clean.groupby(["grade", "gender"])["marks"].mean().unstack().plot(kind="bar")
plt.title("Average Marks by Grade and Gender")
plt.xlabel("Grade")
plt.ylabel("Average Marks")
plt.xticks(rotation=0)
plt.legend(title="Gender")
plt.show()

**Explanation:** For every grade we see one bar for each gender. We can now compare males and females inside each grade.

## Part 5: Line Plots

### What does it show?

A line plot connects points with a line. It shows **how a value changes in order**, for example over time (days, months, years).

### When do we use it?

Only when the data has a **natural order**. Our dataset has no dates, but the students are listed in order of `student_id`, so we can use this order to learn the idea.

**Do not use a line plot for categories with no order** (like section or gender). A line suggests that one point is followed by the next, which is not true for such categories.

### 5.1 Simple line plot of marks

In [ ]:
df_clean["marks"].plot(kind="line")
plt.title("Marks of Students (in order of rows)")
plt.xlabel("Student number (row)")
plt.ylabel("Marks")
plt.show()

**Explanation:** The x-axis is the row number (0 to 99) and the y-axis is the marks. The line jumps up and down a lot because each student is different.

### 5.2 Running average (rolling mean)

A **running average** replaces each point with the average of the last few points. It removes the jumps and shows the **trend**.

In [ ]:
df_clean["marks"].rolling(window=10).mean().plot(kind="line")
plt.title("Running Average of Marks (window of 10 students)")
plt.xlabel("Student number (row)")
plt.ylabel("Average Marks")
plt.show()

**Explanation**

- `rolling(window=10)` looks at 10 students at a time.
- `.mean()` takes the average of those 10 students.
- The first 9 points are empty because there are not yet 10 students before them.
- The line is smoother than the previous plot.

### 5.3 Raw marks and running average together

In [ ]:
df_clean["marks"].plot(kind="line", label="Marks", alpha=0.5)
df_clean["marks"].rolling(window=10).mean().plot(kind="line", label="Running average", color="red")

plt.title("Marks and Running Average")
plt.xlabel("Student number (row)")
plt.ylabel("Marks")
plt.legend()
plt.show()

**Explanation:** `alpha=0.5` makes the first line lighter. The red line shows the trend on top of the raw marks.

## Part 6: Scatter Plots

### What does it show?

A scatter plot shows the **relationship between two continuous variables**. Each dot is one student.

### How to read it

| Pattern of dots | Meaning |
|---|---|
| Going **up** from left to right | Positive relationship |
| Going **down** from left to right | Negative relationship |
| No direction | No clear relationship |

### 6.1 Basic scatter plot

In [ ]:
df_clean.plot(kind="scatter", x="study_hours", y="marks")
plt.title("Study Hours vs Marks")
plt.xlabel("Study Hours")
plt.ylabel("Marks")
plt.show()

**Explanation:** `x` is the variable on the horizontal axis and `y` is on the vertical axis. We want to know: do more study hours lead to more marks?

In [ ]:
df_clean.plot(kind="scatter", x="attendance", y="marks")
plt.title("Attendance vs Marks")
plt.xlabel("Attendance")
plt.ylabel("Marks")
plt.show()

### 6.2 Adding a trend line

A trend line is a straight line that shows the **overall direction** of the dots.

In [ ]:
sns.regplot(data=df_clean, x="study_hours", y="marks")
plt.title("Study Hours vs Marks with Trend Line")
plt.xlabel("Study Hours")
plt.ylabel("Marks")
plt.show()

**Explanation**

- `sns.regplot()` draws the dots and the trend line together.
- The light colour band around the line shows how sure we are about the line.
- If the line goes up, more study hours go with more marks.

In [ ]:
sns.regplot(data=df_clean, x="ai_usage_hours", y="marks")
plt.title("AI Usage Hours vs Marks with Trend Line")
plt.xlabel("AI Usage Hours")
plt.ylabel("Marks")
plt.show()

### 6.3 Using colour to show a third variable (`hue`)

We can colour the dots by a category. Now one plot shows **three** variables.

In [ ]:
sns.scatterplot(data=df_clean, x="study_hours", y="marks", hue="gender")
plt.title("Study Hours vs Marks by Gender")
plt.xlabel("Study Hours")
plt.ylabel("Marks")
plt.show()

**Explanation**

- `hue="gender"` gives a different colour to each gender.
- We can check if the pattern is the same for both groups.

In [ ]:
sns.scatterplot(data=df_clean, x="attendance", y="marks", hue="section")
plt.title("Attendance vs Marks by Section")
plt.xlabel("Attendance")
plt.ylabel("Marks")
plt.show()

## Part 7: Pie Charts

### What does it show?

A pie chart shows **how a whole is divided** into parts. Each slice is the share of one category.

### When does it work?

- When there are **few categories** (2 to 5).
- When we want to show **parts of a whole**.

### When does it not work?

- When there are **many categories**. Small slices are hard to compare. Use a bar chart instead.

In [ ]:
df_clean["grade"].value_counts().sort_index().plot(kind="pie", autopct="%1.1f%%")
plt.title("Grade Distribution")
plt.ylabel("")
plt.show()

**Explanation**

- `kind="pie"` draws the pie chart.
- `autopct="%1.1f%%"` writes the percent on each slice.
- `plt.ylabel("")` removes the extra label.

In [ ]:
df_clean["gender"].value_counts().plot(kind="pie", autopct="%1.1f%%")
plt.title("Gender Share")
plt.ylabel("")
plt.show()

### Pie chart vs bar chart for the same data

Look at `ai_tool`. It has more categories. Compare the two plots.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df_clean["ai_tool"].value_counts().plot(kind="pie", autopct="%1.0f%%", ax=axes[0])
axes[0].set_title("Pie Chart")
axes[0].set_ylabel("")

df_clean["ai_tool"].value_counts().plot(kind="bar", ax=axes[1])
axes[1].set_title("Bar Chart")
axes[1].set_xlabel("AI Tool")
axes[1].set_ylabel("Number of Students")

plt.show()

**Lesson:** With many categories, it is much easier to compare the bars than the slices. So we prefer a bar chart.

## Part 8: Pair Plot and Correlation Heatmap

When we have many continuous variables, we want to see **all relationships together**.

### 8.1 Pair plot

A pair plot draws a **scatter plot for every pair** of variables. On the diagonal it shows the histogram of each variable.

In [ ]:
sns.pairplot(df_clean[["study_hours", "attendance", "ai_usage_hours", "marks"]])
plt.show()

**Explanation**

- We give it 4 continuous columns. It makes a 4 x 4 grid of small plots.
- Each small plot compares two variables. The diagonal shows one variable alone.
- To check what affects marks, look at the **marks row** (or column) and see which plots show a clear upward or downward pattern.
- We did not include more columns because too many columns make the grid hard to read.

### Pair plot with colour

In [ ]:
sns.pairplot(df_clean[["study_hours", "attendance", "ai_usage_hours", "marks", "gender"]], hue="gender")
plt.show()

**Explanation:** `hue="gender"` colours the dots by gender in every small plot.

### 8.2 Correlation heatmap

**Correlation** is one number between -1 and +1 that tells how strongly two variables move together.

| Correlation | Meaning |
|---|---|
| Close to **+1** | Strong positive: when one goes up, the other goes up |
| Close to **0** | No linear relationship |
| Close to **-1** | Strong negative: when one goes up, the other goes down |

A **heatmap** shows the correlation of every pair of variables as colours, so we can see everything at a glance.

First, the correlation table:

In [ ]:
corr = df_clean[continuous_cols].corr()
corr

Now the heatmap:

In [ ]:
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

**Explanation**

- `sns.heatmap(corr)` turns the table into colours.
- `annot=True` writes the number inside every box.
- `cmap="coolwarm"` means red for positive and blue for negative.
- The diagonal is always 1, because a variable is perfectly related to itself.

### How to read it

- Look at the **marks** row. Which box is the **reddest** (strongest positive)? Which is the bluest (negative)?
- A value near 0 means little or no straight-line relationship.

### Remember

**Correlation does not mean causation.** If two variables move together, it does not prove that one causes the other.

In [ ]:
corr["marks"].sort_values(ascending=False)

**Explanation:** This shows the correlation of every variable with marks, from the highest to the lowest. Marks with itself is always 1 and is at the top.

## Part 9: Choosing the Right Plot

### Decision table: question to plot

| My question | Variable type | Best plot |
|---|---|---|
| What is the spread and are there outliers? | One continuous | Box plot |
| What is the shape of the data? | One continuous | Histogram |
| How many in each category? | One categorical | Count plot (bar of `value_counts`) |
| Compare an average across categories | Categorical and continuous | Bar chart |
| Compare spread across categories | Categorical and continuous | Grouped box plot |
| How does a value change in order or time? | Ordered data | Line plot |
| Are two numbers related? | Two continuous | Scatter plot |
| What share does each part have? (few parts) | One categorical | Pie chart |
| All relationships between many numbers | Many continuous | Pair plot |
| Strength of relationships at a glance | Many continuous | Correlation heatmap |

### Common mistakes to avoid

1. **Pie chart with too many slices.** After about 5 categories, use a bar chart.
2. **Forgetting labels.** Every plot needs a title, x label and y label.
3. **Line plot for unordered categories.** A line implies order.
4. **Cutting the y-axis in a bar chart.** It exaggerates small differences.
5. **Thinking correlation means causation.**

### Mistake 4 in action

The two plots below show the **same data**. Look at how different they feel.

In [ ]:
avg_marks = df_clean.groupby("section")["marks"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: y-axis cut (misleading)
avg_marks.plot(kind="bar", ax=axes[0])
axes[0].set_ylim(avg_marks.min() - 2, avg_marks.max() + 1)
axes[0].set_title("Misleading: y-axis is cut")
axes[0].set_xlabel("Section")
axes[0].set_ylabel("Average Marks")

# Plot 2: y-axis starts at 0 (honest)
avg_marks.plot(kind="bar", ax=axes[1])
axes[1].set_ylim(0, 100)
axes[1].set_title("Honest: y-axis starts at 0")
axes[1].set_xlabel("Section")
axes[1].set_ylabel("Average Marks")

plt.show()

**Explanation**

- `set_ylim(low, high)` sets the range of the y-axis.
- In the left plot the bars look very different, but the real difference is small.
- In the right plot, where the y-axis starts at 0, we see the true picture.

### Hands-on practice: choose and build the plot

**Question 1.** How many students are in each year?

**Question 2.** Are there any outliers in `ai_usage_hours`?

**Question 3.** Do students who attend more also score more marks?

**Question 4.** What is the average study hours for each grade?

**Question 5.** What does the distribution of `age` look like?

Try each one yourself first. The answers are in the cells below.

### Answers

**Answer 1:** Count plot

In [ ]:
df_clean["year"].value_counts().sort_index().plot(kind="bar")
plt.title("Number of Students in Each Year")
plt.xlabel("Year")
plt.ylabel("Number of Students")
plt.xticks(rotation=0)
plt.show()

**Answer 2:** Box plot

In [ ]:
df_clean["ai_usage_hours"].plot(kind="box")
plt.title("Box Plot of AI Usage Hours")
plt.ylabel("AI Usage Hours")
plt.show()

**Answer 3:** Scatter plot with a trend line

In [ ]:
sns.regplot(data=df_clean, x="attendance", y="marks")
plt.title("Attendance vs Marks")
plt.xlabel("Attendance")
plt.ylabel("Marks")
plt.show()

**Answer 4:** Bar chart of averages

In [ ]:
df_clean.groupby("grade")["study_hours"].mean().plot(kind="bar")
plt.title("Average Study Hours by Grade")
plt.xlabel("Grade")
plt.ylabel("Average Study Hours")
plt.xticks(rotation=0)
plt.show()

**Answer 5:** Histogram

In [ ]:
df_clean["age"].plot(kind="hist", bins=8)
plt.title("Histogram of Age")
plt.xlabel("Age")
plt.ylabel("Number of Students")
plt.show()

## Part 10: Wrap-Up and Practice

### Summary of all plots

| Plot | Code |
|---|---|
| Box plot | `df["col"].plot(kind="box")` |
| Box plot by group | `df.boxplot(column="num", by="cat")` |
| Histogram | `df["col"].plot(kind="hist", bins=10)` |
| Count plot | `df["col"].value_counts().plot(kind="bar")` |
| Bar chart (average) | `df.groupby("cat")["num"].mean().plot(kind="bar")` |
| Grouped bar chart | `df.groupby(["a", "b"])["num"].mean().unstack().plot(kind="bar")` |
| Line plot | `df["col"].plot(kind="line")` |
| Running average | `df["col"].rolling(window=10).mean().plot(kind="line")` |
| Scatter plot | `df.plot(kind="scatter", x="a", y="b")` |
| Scatter with trend line | `sns.regplot(data=df, x="a", y="b")` |
| Scatter with colour | `sns.scatterplot(data=df, x="a", y="b", hue="cat")` |
| Pie chart | `df["col"].value_counts().plot(kind="pie", autopct="%1.1f%%")` |
| Pair plot | `sns.pairplot(df[list_of_cols])` |
| Correlation heatmap | `sns.heatmap(df[cols].corr(), annot=True, cmap="coolwarm")` |

### Useful extra commands

| Task | Code |
|---|---|
| Title | `plt.title("...")` |
| x-axis label | `plt.xlabel("...")` |
| y-axis label | `plt.ylabel("...")` |
| Rotate x labels | `plt.xticks(rotation=30)` |
| Show legend | `plt.legend()` |
| Many plots in a row | `fig, axes = plt.subplots(1, 3, figsize=(15, 4))` |
| Display the plot | `plt.show()` |

### Practice Exercises

For each exercise: **choose the plot, build it, and write one line about what it shows.**

**Easy**

1. Draw a box plot of `study_hours`.
2. Draw a histogram of `attendance` with 10 bins.
3. Draw a count plot of `gender`.

**Medium**

4. Draw a box plot of `marks` grouped by `year`.
5. Draw a bar chart of average `ai_usage_hours` for each `section`.
6. Draw a scatter plot of `study_hours` vs `attendance`.

**Challenging**

7. Draw a grouped bar chart of average `marks` by `section` and `gender`.
8. Draw a scatter plot of `ai_usage_hours` vs `marks` coloured by `year`.
9. Draw a correlation heatmap and write which two variables (apart from each variable with itself) have the strongest relationship.
10. Draw a pie chart of `complaint`. Then draw a bar chart for the same data. Which one is easier to read?

### Next session

In the next session we continue building on what we have learned about the data and the plots.